# 01 -- Variational Autoencoders

**Goal:** learn a *latent space* -- a low-dimensional code `z` for every image -- and a decoder
that turns any point of that space into an image. Sampling is then: draw `z ~ N(0, I)`, decode.

**In this notebook:** the ELBO from first principles, the reparameterisation trick, training on
MNIST or your Ray_dog photos, reconstructions, latent interpolations, and pushing the model to
the Hugging Face Hub.

*Paper:* Kingma & Welling, *Auto-Encoding Variational Bayes* (2013).

## 1. Theory

We want a model $p_\theta(x) = \int p_\theta(x\mid z)\,p(z)\,dz$ with a simple prior $p(z)=\mathcal N(0, I)$
and a neural decoder $p_\theta(x\mid z)$. The integral over $z$ is intractable, so we introduce an
**encoder** $q_\phi(z\mid x)=\mathcal N(\mu_\phi(x), \operatorname{diag}\sigma^2_\phi(x))$ that guesses which
codes could have produced $x$. Jensen's inequality gives the **evidence lower bound (ELBO)**:

$$
\log p_\theta(x) \;\ge\; \underbrace{\mathbb E_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]}_{\text{reconstruction}}
\;-\; \underbrace{\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big)}_{\text{stay close to the prior}}
$$

* With a Gaussian decoder, $-\log p_\theta(x\mid z)$ is (up to constants) the **squared error** between $x$ and its reconstruction.
* The KL between two Gaussians has a closed form: $\tfrac12\sum_j\big(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1\big)$.

**Reparameterisation trick.** We cannot backpropagate through "sample $z$". Writing
$z = \mu + \sigma\odot\varepsilon,\ \varepsilon\sim\mathcal N(0,I)$ moves the randomness into $\varepsilon$,
so gradients flow into $\mu$ and $\sigma$.

**$\beta$-VAE.** Loss $=\text{recon}+\beta\,\text{KL}$. $\beta>1$ forces a more factorised, more "disentangled"
latent space at the price of blurrier images. Setting `method.beta` in the config switches between the two.

**Why VAE samples are blurry.** The squared-error likelihood rewards predicting the *mean* of all
plausible images for a code; averages of sharp images are blurry. GANs, diffusion and flow matching
(the next notebooks) do not have this problem -- which is a large part of why they exist.

**KL warm-up.** Early in training the KL term can win and the encoder ignores the input
("posterior collapse"). `method.kl_warmup_steps` ramps $\beta$ from 0.

## 0. Setup

Run from the repo root or from `notebooks/`. The package is imported from the repo,
so editing `genmodels/*.py` and re-running a cell picks up the change after a kernel restart.

On **RunPod**: `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
On **Colab**: uncomment the first line.

In [ ]:
# !git clone -q https://github.com/yar78/vision-generative-models-survey.git && %cd vision-generative-models-survey && pip install -q -e .
import sys, pathlib
REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.vae import VAE, vae_loss
print("device:", get_device())

### The loss, as implemented

In [ ]:
import inspect; print(inspect.getsource(vae_loss))

## 2. Choose an experiment

Every experiment is one YAML file in `configs/`, shared with the command-line script,
so a run started here and a run started on a RunPod GPU are the same experiment.

* `configs/vae/mnist.yaml` -- the quick default.
* `configs/ray_dog/vae.yaml` -- your **Ray_dog** photos (`yaryar78/Ray_dog`, 258 images) at 64x64.

Edit `cfg` below to change anything (steps, batch size, model width ...).

In [ ]:
CONFIG = REPO / "configs/vae/mnist.yaml"        # or: REPO / "configs/ray_dog/vae.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 5000

print(yaml.safe_dump(cfg, sort_keys=False))

### Look at the training data (with the training-time augmentation applied)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:64].tolist()])
show_grid(x_real, title="training data")

## 3. Build the model

`build_trainer(cfg)` creates the VAE (encoder + decoder), the optimizers and the training loop.
Outputs go to `outputs/<name>/` (on RunPod: `/workspace/genmodels/outputs/<name>/`, on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.2f}M params" for k, v in trainer.num_params().items()})

In [ ]:
print(trainer.model)

## 4. Train

`trainer.fit()` runs `train.steps` steps. Every `sample_every` steps it saves a fixed-seed
sample grid to `samples/`; every `ckpt_every` steps a checkpoint. Re-running the cell
(or the script) **resumes** from the checkpoint.

MNIST: ~5 min on a GPU for 20k steps, a few minutes on CPU for ~2k steps (`cfg['train']['steps'] = 2000`). Watch `recon` fall and `kl` rise as the KL warm-up ends.

Already trained this config with the script (e.g. on RunPod)? Set `TRAIN = False`
and the checkpoint in the run directory is loaded instead.

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### TensorBoard
Loss curves and sample grids over time. On RunPod, TensorBoard is also served on port 6006 by `scripts/runpod/train.sh`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
plot_history(trainer.history)

## 5. Results

### Random samples: z ~ N(0, I) -> decoder

In [ ]:
show_grid(trainer.sample_fixed(64), title='samples')

### Reconstructions: top = input, bottom = decode(encode(x))
Reconstructions are always sharper than samples: the decoder only has to fill in what the code missed.

In [ ]:
x = x_real[:16]
show_grid(torch.cat([x, trainer.reconstruct(x).cpu()]), nrow=16, size=12)

## 6. Exploring the latent space

Walk in a straight line between the codes of two images. In a good VAE every point on the path
decodes to a plausible image -- the latent space is *smooth* because the KL term packs codes
around the origin with no gaps.

In [ ]:
model = trainer.model.eval()
with torch.no_grad():
    mu, _ = model.encode(x_real[:8].to(trainer.device))
    rows = []
    for a, b in zip(mu[0::2], mu[1::2]):
        alphas = torch.linspace(0, 1, 10, device=trainer.device)[:, None]
        rows.append(model.decode((1 - alphas) * a + alphas * b))
show_grid(torch.cat(rows), nrow=10, title="latent interpolations", size=10)

### What does each latent dimension do?
Vary one coordinate of a fixed code from -3 to +3 while keeping the others fixed. With $\beta>1$ more dimensions control a single, interpretable factor.

In [ ]:
with torch.no_grad():
    z0 = torch.zeros(1, model.latent_dim, device=trainer.device)
    vals = torch.linspace(-3, 3, 9, device=trainer.device)
    dims = range(min(8, model.latent_dim))
    rows = []
    for d in dims:
        z = z0.repeat(len(vals), 1)
        z[:, d] = vals
        rows.append(model.decode(z))
show_grid(torch.cat(rows), nrow=9, title="latent traversals (one row per dimension)", size=9)

## 7. Push to the Hugging Face Hub

`push_run` uploads the run directory as a model repo **`<your-username>/{cfg['name']}`**
(private by default):

| file | |
|---|---|
| `model.safetensors`, `config.json` | weights + constructor args (`VAE.from_pretrained` reads these) |
| `README.md` | generated model card: dataset, steps, hardware, git commit, metrics |
| `samples.png`, `metrics.json`, `train_config.yaml` | results and the exact config |
| `runs/` | TensorBoard logs (shown in the Hub's *Training metrics* tab) |

Login once with a **write** token from https://huggingface.co/settings/tokens
(`notebook_login()` below, `huggingface-cli login`, or the `HF_TOKEN` env var). To use another
repo name set `cfg["hub"]["repo_id"]`; to publish, `cfg["hub"]["private"] = False` before training,
or `push_run(..., private=False)`.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Reload it from the Hub
This is how anyone (including future you) uses the model.

In [ ]:
if PUSH_TO_HUB:
    from genmodels.vae import VAE
    model = VAE.from_pretrained(resolve_repo_id(cfg)).to(trainer.device).eval()
    show_grid(model.generate(16), title="reloaded from the Hub")

## 8. Things to try

* `method.beta = 4` (β-VAE): compare traversals -- are the factors cleaner? Are samples blurrier?
* `model.latent_dim = 2` on MNIST: decode a grid over $[-3,3]^2$ to *see* the whole latent space.
* Ray_dog: how much detail survives at 64x64? This blur is what latent diffusion's VQ/KL autoencoder fixes with a perceptual + adversarial loss.